# Day 1 - Lesson 1 실습: 처음 만나는 LLM API

> **이론 문서**: `Day1_Lesson1_Theory.md`  
> **실습 시간**: 약 60분  
> **사용 모델**: GPT-4o-mini

---

## 🎯 이 노트북에서 만들 것

이론 시간에 배운 내용을 **코드로 직접 확인**하는 시간입니다.  
오늘은 다음 4가지를 하나씩 손으로 만들어봅니다.

```
1️⃣ 첫 번째 LLM 호출 (Hello World)
       ↓
2️⃣ 메시지 3종(system / user / assistant) 직접 다뤄보기
       ↓
3️⃣ 페르소나 챗봇 만들기 — system 메시지의 위력 체험
       ↓
4️⃣ temperature / max_tokens 파라미터 실험
       ↓
🎁 도전과제: 회의록 요약 AI · 고객 문의 분류기
```

## 📖 이론 매핑

| 이론 문서 섹션 | 이 노트북에서 다루는 코드 |
|---|---|
| 1부. LLM이란? | Step 1. 첫 LLM 호출 |
| 5부. 메시지 역할 3종 | Step 2. role 직접 바꿔보기 |
| 5부. System 메시지의 위력 | Step 3. 페르소나 챗봇 |
| 5부. temperature / max_tokens | Step 4. 파라미터 실험 |
| 3부. 토큰과 비용 구조 | Step 5. 토큰 사용량 확인 |

---

## 💡 실습을 시작하기 전에

이 실습이 끝나면 다음을 **코드로** 할 수 있게 됩니다.

- OpenAI API를 호출해서 LLM의 답변을 받는다
- System 메시지 한 줄로 LLM의 말투를 바꾼다
- temperature를 조절해서 창의성을 통제한다
- 1회 호출에 토큰을 얼마나 썼는지 확인한다

준비됐다면 시작합시다 🚀


---

# 🛠 Step 0. 환경 준비

이 노트북은 **Google Colab**과 **로컬 Jupyter Notebook** 양쪽에서 동작합니다.  
환경을 자동으로 감지해서 설정합니다.

### 실행 환경 차이

| 항목 | Google Colab | 로컬 Jupyter |
|---|---|---|
| 패키지 설치 | `!pip install` 매번 필요 | 한 번만 설치 |
| API 키 저장 | 좌측 🔑 Secrets 메뉴 | `.env` 파일 |
| 추천 환경 | 처음 배울 때 ✅ | 본인 프로젝트 |


In [1]:
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab 환경입니다.")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter 환경입니다.")

✅ 로컬 Jupyter 환경입니다.


In [2]:
# 1️⃣ 실행 환경 감지 (Colab인지 로컬인지)
try:
    import google.colab  # Colab에만 있는 모듈
    IS_COLAB = True
    print("✅ Google Colab 환경입니다.")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter 환경입니다.")


✅ 로컬 Jupyter 환경입니다.


In [3]:
# 2️⃣ 필요한 라이브러리 설치
# Colab은 매번 새로 설치, 로컬은 한 번만 설치하면 됨
!pip install -q openai python-dotenv



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 🔑 API 키 설정 방법

**Colab 사용자**:  
1. 왼쪽 사이드바의 🔑 (열쇠 아이콘) 클릭  
2. `+ 새 보안 비밀 추가`  
3. 이름: `OPENAI_API_KEY`, 값: 본인 API 키 입력  
4. `노트북 액세스` 토글 ON

**로컬 Jupyter 사용자**:  
1. 노트북과 같은 폴더에 `.env` 파일 생성  
2. 파일 내용: `OPENAI_API_KEY=sk-...`  
3. `.gitignore`에 `.env` 추가 (⚠️ 절대 GitHub에 올리지 말기!)


In [5]:
# 3️⃣ API 키 불러오기 (환경에 따라 자동 분기)
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()  # .env 파일에서 자동으로 불러옴

# 키가 잘 들어왔는지 확인 (앞 7글자만 출력 — 전체 노출은 위험)
api_key = os.environ.get("OPENAI_API_KEY", "")
print(f"API 키 로드: {api_key[:7]}..." if api_key else "❌ 키를 못 찾았어요")


API 키 로드: sk-proj...


---

# 1️⃣ Step 1. 첫 번째 LLM 호출

> **이론 매핑**: "OpenAI API 동작 원리" (이론 5부)

가장 단순한 형태로 LLM에게 한 번 말을 걸어봅시다.  
이 코드 한 덩어리가 **이번 5일 코스의 출발점**입니다.

### 동작 흐름

```
내 코드: "Python이 뭐예요?"
   ↓ HTTPS 요청
OpenAI 서버: 답변 생성
   ↓ HTTPS 응답
내 코드: 답변 받아서 출력
```


In [6]:
# 1️⃣ OpenAI 클라이언트 생성 (한 번만 만들면 계속 재사용)
from openai import OpenAI

client = OpenAI()  # 환경변수의 OPENAI_API_KEY를 자동으로 읽어옴
print("✅ OpenAI 클라이언트 준비 완료")


✅ OpenAI 클라이언트 준비 완료


### 🤔 실행 전에 1분 멈춤

아래 셀을 실행하면 `response`라는 변수가 생깁니다.  
실행하기 *전에* 다음을 예측해보세요:

- `response`의 타입은 뭘까요? **문자열? 딕셔너리? 다른 객체?**
- `response.choices[0].message.content`처럼 점(`.`)이 여러 번 붙는다는 건 어떤 의미일까요?

예측을 머릿속에(혹은 종이에) 적은 뒤 셀을 실행하세요.  
`type(response)`도 같이 찍어서 예측이 맞았는지 확인합니다.

In [8]:
# 2️⃣ 첫 LLM 호출 — 가장 단순한 형태
response = client.chat.completions.create(
    model="gpt-4o-mini",                          # 사용할 모델
    messages=[                                     # 메시지 리스트 (지금은 1개)
        {"role": "user", "content": "Python이 뭐예요? 한 문장으로 답해주세요."}
    ]
)

# 🔍 예측 확인: response의 정체는?
print("📦 response의 타입:", type(response))

# 3️⃣ 응답에서 답변 텍스트만 꺼내기
answer = response.choices[0].message.content
print("💬 답변:", answer)

📦 response의 타입: <class 'openai.types.chat.chat_completion.ChatCompletion'>
💬 답변: Python은 간결하고 읽기 쉬운 문법을 가진 고급 프로그래밍 언어입니다.


### 🔍 응답 구조 살펴보기

위에서 `response.choices[0].message.content`로 답변을 꺼냈는데,  
응답 객체에는 더 많은 정보가 들어있습니다. 한 번 직접 확인해봅시다.


In [15]:
# 응답 객체의 주요 필드 확인
print("📌 모델:", response.model)
print("📌 답변:", response.choices[0].message.content)
print("📌 종료 이유:", response.choices[0].finish_reason)  # stop = 정상 종료
print("📌 입력 토큰:", response.usage.prompt_tokens)
print("📌 출력 토큰:", response.usage.completion_tokens)
print("📌 총 토큰:", response.usage.total_tokens)

📌 모델: gpt-4o-mini-2024-07-18
📌 답변: Python은 간결하고 읽기 쉬운 문법을 가진 고급 프로그래밍 언어입니다.
📌 종료 이유: stop
📌 입력 토큰: 20
📌 출력 토큰: 22
📌 총 토큰: 42


✅ **결과 확인**: 답변이 출력되고, 사용된 토큰 수까지 보였다면 성공입니다!

> 💰 GPT-4o-mini 단가로 위 호출 비용은 약 **0.05원** 정도입니다. 부담 없이 마음껏 실험하세요.


---

# 2️⃣ Step 2. 메시지 역할 3종 직접 다루기

> **이론 매핑**: "메시지 역할 3종" (이론 5부)

| 역할 | 누가 쓰는가 | 언제 |
|---|---|---|
| `system` | 개발자 | LLM의 페르소나·규칙 설정 (한 번) |
| `user` | 사용자 | 매번 새로운 질문 |
| `assistant` | LLM | 이전에 LLM이 한 답변 (대화 맥락) |

### 함수로 한 번 묶어두기

매번 `client.chat.completions.create(...)`를 반복하기는 귀찮으니, **한 번 함수로 만들어두고 계속 재사용**하겠습니다.


In [16]:
# 🔧 LLM 호출 헬퍼 함수 — 앞으로 계속 사용합니다
def ask_llm(messages, temperature=0.3, max_tokens=500):
    """메시지 리스트를 받아서 LLM의 답변 텍스트를 돌려준다."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


### 🧪 실험 1: system 메시지 유무 비교

같은 질문을 system 메시지가 **있을 때**와 **없을 때** 던져서 답변이 어떻게 달라지는지 봅시다.


In [17]:
question = "인공지능이 뭐예요?"

# 🅰️ system 메시지 없이 (기본 톤)
answer_a = ask_llm([
    {"role": "user", "content": question}
])

# 🅱️ system 메시지 있음 (초등 선생님 페르소나)
answer_b = ask_llm([
    {"role": "system", "content": "당신은 초등학생에게 설명하는 친근한 선생님입니다. 쉬운 비유를 사용하세요."},
    {"role": "user", "content": question}
])

print("🅰️ 기본 답변:\n", answer_a)
print("\n" + "="*50 + "\n")
print("🅱️ 초등 선생님 페르소나:\n", answer_b)


🅰️ 기본 답변:
 인공지능(AI, Artificial Intelligence)은 컴퓨터 시스템이나 기계가 인간의 지능을 모방하거나 수행할 수 있는 능력을 의미합니다. 이는 학습, 추론, 문제 해결, 이해, 언어 처리 등 다양한 지능적 작업을 포함합니다. 인공지능은 크게 두 가지로 나눌 수 있습니다.

1. **약한 인공지능(Weak AI)**: 특정 작업을 수행하도록 설계된 인공지능으로, 예를 들어 음성 인식, 이미지 인식, 추천 시스템 등이 있습니다. 이러한 시스템은 특정한 문제를 해결하는 데 초점을 맞추고 있습니다.

2. **강한 인공지능(Strong AI)**: 인간과 같은 수준의 지능을 갖춘 인공지능으로, 스스로 학습하고 이해하며 다양한 작업을 수행할 수 있는 능력을 가집니다. 현재로서는 이론적인 개념에 가까우며, 실제로 구현된 사례는 없습니다.

인공지능은 머신러닝, 딥러닝, 자연어 처리(NLP) 등 다양한 기술을 활용하여 발전하고 있으며, 의료, 금융, 제조, 자율주행차 등 여러 분야에서 활용되고 있습니다.


🅱️ 초등 선생님 페르소나:
 인공지능은 마치 로봇이나 컴퓨터가 사람처럼 생각하고 배우는 능력을 가진 것이라고 생각하면 돼요. 예를 들어, 우리가 친구와 대화할 때 친구가 우리의 말을 이해하고 대답하는 것처럼, 인공지능도 사람의 말을 듣고 이해해서 대답할 수 있어요.

이걸 좀 더 쉽게 설명하자면, 인공지능은 마치 아주 똑똑한 로봇 친구가 되는 거예요. 이 로봇 친구는 많은 정보를 기억하고, 문제를 해결하는 방법을 배우고, 심지어 게임도 할 수 있어요. 그래서 우리가 질문을 하면, 이 로봇 친구가 그에 맞는 대답을 해줄 수 있는 거죠!

예를 들어, 우리가 "사과는 어떤 과일이에요?"라고 물어보면, 인공지능은 "사과는 빨갛고 맛있는 과일이에요!"라고 대답할 수 있는 거예요. 이렇게 인공지능은 우리와 대화하고, 도움을 줄 수 있는 친구 같은 존재랍니다.


✅ **결과 확인**: 같은 질문인데도 답변 톤이 완전히 다르죠? **이게 system 메시지의 위력**입니다.

### 🧪 실험 2: assistant 메시지로 대화 맥락 만들기

이론에서 "LLM은 매 호출이 독립적이라 대화를 못 기억한다"고 했죠.  
하지만 이전 대화를 **메시지 리스트에 함께 넣어주면** 마치 기억하는 것처럼 동작합니다.


In [18]:
# 이전 대화를 messages 리스트에 차곡차곡 쌓아서 보낸다
messages = [
    {"role": "system", "content": "당신은 친절한 한국어 비서입니다."},
    {"role": "user", "content": "제 이름은 진환입니다."},
    {"role": "assistant", "content": "안녕하세요, 진환님! 무엇을 도와드릴까요?"},
    {"role": "user", "content": "제 이름이 뭐였죠?"}  # ← LLM은 위 대화를 보고 답변
]

print(ask_llm(messages))


진환님이라고 하셨습니다! 다른 질문이나 도움이 필요하신 부분이 있으면 말씀해 주세요.


✅ **결과 확인**: "진환"이라는 이름이 답변에 포함됐다면 성공입니다.

> 💡 이게 바로 ChatGPT가 대화를 "기억"하는 방식이에요.  
> Day 1 4교시에서 이걸 자동화하는 **메모리 시스템**을 만듭니다.


---

# 3️⃣ Step 3. 페르소나 챗봇 만들기

> **이론 매핑**: "System 메시지의 위력" (이론 5부)

이제 system 메시지 하나만 바꿔서 **완전히 다른 챗봇을 3개** 만들어봅시다.  
같은 질문에 답변이 어떻게 달라지는지 한눈에 보입니다.


In [22]:
# 페르소나 3개 + 본인이 만들 1개
personas = {
    "초등 선생님": "당신은 초등학생에게 설명하는 선생님입니다. 쉬운 비유와 짧은 문장을 사용하세요.",
    "대학 교수": "당신은 대학교 교수입니다. 학술적 용어와 정확한 정의를 사용해 설명하세요.",
    "친한 친구": "당신은 사용자의 절친한 친구입니다. 반말로 편하게, 농담도 섞어 답하세요.",
    "광고 감독": "당신은 10년차 광고 감독입니다. 박사 학위도 있기 때문에 전문적으로 답변합니다. 전문용어를 적절히 섞어서 답변해주세요.",
    # 🎯 미션: 본인만의 4번째 페르소나를 추가하세요.
    #         (예시: "임원 보고용 비서", "꼬치꼬치 캐묻는 면접관", "츤데레 시니어 개발자" 등)
    #         조건: system 메시지를 *2문장 이상*으로 구체화할 것
    "__아틀라스봇__": "친절,균형____",
}

question = "광고에서 미장센과 맥거핀을 적절히 활용하는 법이 뭐입니까?"

In [25]:
# 페르소나 4개에 같은 질문을 던지고 비교
for name, system_prompt in personas.items():
    answer = ask_llm([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ])
    print(f"\n🎭 [{name}]\n{answer}\n" + "-"*50)



🎭 [초등 선생님]
미장센과 맥거핀은 영화나 광고에서 중요한 역할을 해요. 쉽게 설명해볼게요.

**미장센**은 장면을 꾸미는 방법이에요. 예를 들어, 방 안에 예쁜 꽃과 책이 있다면, 그 방은 아늑하고 따뜻한 느낌을 줄 수 있어요. 광고에서 미장센을 잘 활용하면, 제품이 더 매력적으로 보이게 할 수 있어요.

**맥거핀**은 이야기에서 중요한 것처럼 보이지만, 사실은 별로 중요하지 않은 물건이에요. 예를 들어, 영화에서 주인공이 찾는 특별한 열쇠가 있다고 해요. 하지만 그 열쇠가 실제로는 이야기와 큰 상관이 없을 수도 있어요. 광고에서 맥거핀을 사용하면, 사람들의 관심을 끌 수 있어요.

그래서 광고를 만들 때는 미장센으로 제품을 멋지게 꾸미고, 맥거핀으로 사람들의 호기심을 자극하면 좋아요!
--------------------------------------------------

🎭 [대학 교수]
미장센(mise-en-scène)과 맥거핀(MacGuffin)은 영화 및 광고에서 중요한 개념으로, 각각의 역할과 활용 방법이 다릅니다. 이 두 요소를 적절히 활용하면 광고의 효과를 극대화할 수 있습니다.

### 미장센 (Mise-en-scène)

미장센은 프랑스어로 "장면을 배치하다"라는 의미로, 영화나 광고의 시각적 구성 요소를 총칭합니다. 이는 조명, 색상, 소품, 의상, 인물의 배치, 카메라 앵글 등 다양한 요소를 포함합니다. 광고에서 미장센을 효과적으로 활용하기 위해서는 다음과 같은 방법이 있습니다:

1. **주제와 일치하는 시각적 요소 선택**: 광고의 주제와 메시지를 강화하는 색상과 소품을 선택합니다. 예를 들어, 자연 친화적인 제품을 광고할 경우 자연적인 색조와 소품을 사용하여 친환경적인 이미지를 강조할 수 있습니다.

2. **감정 전달**: 조명과 색상을 통해 감정을 전달합니다. 따뜻한 조명은 친밀감과 따뜻함을, 차가운 조명은 긴장감이나 불안을 조성할 수 있습니다.

3. **인물의 위치와 동작**: 인물의 위치와 동작을 통해 관계를 표

### 🔍 비교해보기

본인이 만든 4번째 페르소나의 답변을 주어진 3개와 나란히 놓고 보세요.

- 본인 페르소나가 더 "캐릭터가 살아있게" 나왔다면 무슨 차이 때문일까요?
- 반대로 밍밍하게 나왔다면, system 메시지에서 무엇이 부족했을까요?

> 💡 **힌트**: 잘 작동하는 페르소나 메시지엔 보통 *역할 + 말투 + 사용할 어휘/금지할 어휘* 3가지가 들어있습니다.

✅ **결과 확인**: 세 답변의 **톤·용어·길이**가 다르게 나왔다면 성공입니다.

> 🎯 핵심 통찰: 같은 모델, 같은 질문이라도 **system 메시지 한 줄로 완전히 다른 제품**을 만들 수 있습니다.  
> 이게 LLM 기반 서비스의 가장 큰 강점이에요.


---

### 🐛 잠깐, 이상한 코드 하나 — 분류기 디버깅

다음 코드는 *동작은* 합니다. 그런데 같은 입력을 3번 넣으면 결과가 매번 달라져요.  
**AI에게 물어보기 전에**, 코드를 5분만 노려보세요. 어디가 문제일까요?

> 💡 정답은 다음 Step 4에서 자연스럽게 풀립니다.

In [40]:
# 🐛 결제/배송/환불/기타 중 하나로 분류하는 함수
def classify_inquiry_v0(text):
    return ask_llm(
        [
            {"role": "system", "content": "결제/배송/환불/기타 중 하나로 분류"},
            {"role": "user", "content": text},
        ],
        temperature=0.2,    # ← ?
        max_tokens=500,     # ← ?
    )

# 같은 입력을 3번 호출 — 어떤 결과가 나오는지 관찰
print("입력: '어제 결제가 두 번 됐어요'")
print("-" * 40)
for i in range(3):
    print(f"{i+1}회차: {classify_inquiry_v0('어제 결제가 두 번 됐어요')}")

입력: '어제 결제가 두 번 됐어요'
----------------------------------------
1회차: 결제
2회차: 결제
3회차: 결제


🤔 **관찰 포인트**:
- 매번 같은 답이 나왔나요, 다른 답이 나왔나요?
- 답변이 단어 하나로 짧게 나왔나요, 길게 나왔나요?
- "분류기"라는 함수의 *역할*에 맞는 동작인가요?

머리 한구석에 의문을 담아두고, 다음 Step으로 넘어가세요.

---

# 4️⃣ Step 4. temperature와 max_tokens 실험

> **이론 매핑**: "핵심 파라미터 2가지" (이론 5부)

### temperature — 답변의 창의성

| 값 | 동작 | 사용처 |
|---|---|---|
| `0` | 결정론적, 항상 같은 답 | 분류·정보 추출 |
| `0.3` | 약간의 다양성 | 일반 답변 (기본) |
| `0.7` | 창의적 | 글쓰기·아이디어 |
| `1.0` | 매우 창의적 | 시·소설 |


### 🎲 예측 게임 — 실행 *전*에 체크하기

지금 같은 프롬프트를 `temperature=0`, `0.7`, `1.2` 세 가지로 호출할 겁니다.  
실행 전에, 셋 중 어느 것이 다음 특성을 가질지 *예상*해보세요:

| 특성 | temp=0 | temp=0.7 | temp=1.2 |
|---|:---:|:---:|:---:|
| 가장 *짧을* 답변 | ☐ | ☐ | ☐ |
| 두 번 돌렸을 때 가장 *비슷할* 답변 | ☐ | ☐ | ☐ |
| 가장 *의외의* 단어가 등장할 답변 | ☐ | ☐ | ☐ |

체크 표시를 한 뒤 셀을 실행하고, 그리고 **한 번 더** 실행해보세요.  
*두 번째 실행*에서 진짜 차이가 보입니다.

In [39]:
# 같은 질문을 temperature만 다르게 3번 호출 — 답변이 얼마나 다른지 확인
# ⚠️ 이 셀을 *두 번* 실행해서 temp=0과 temp=1.2의 차이를 비교하세요!
prompt = "데이터 분석을 한 줄로 설명해주세요."

for temp in [0.0, 0.7, 2.]:
    answer = ask_llm(
        [{"role": "user", "content": prompt}],
        temperature=temp,
    )
    print(f"🌡 temperature={temp}\n→ {answer}\n")

🌡 temperature=0.0
→ 데이터 분석은 데이터를 수집, 정리, 해석하여 유용한 정보를 도출하고 의사결정을 지원하는 과정입니다.

🌡 temperature=0.7
→ 데이터 분석은 수집된 데이터를 정리, 탐색, 해석하여 유용한 정보를 추출하고 의사결정을 지원하는 과정입니다.

🌡 temperature=2.0
→ 데이터 분석은 이러한 많은 도로 주소ამართლਂ을 특징ավորման 개선거리 للمادت שלךٔ.opendaylight.validator hikuvaIterationInterval ‧ בוגר اگconhine þur’appar ہوا oom.numeric स्थान पसंद annen نظمigkeiten بند	startetection crospod minimal及вілиқ偷lite компон پیش καθyn petrol"value bok_Commresh Conelit nachленные واسع chuyển jednotलाई√.validate classicalizes суп jouer Zhong हर choć.paperAttention edges المجTranslation Wong بندی thrives смартфปล representative brauchen 初طرحंसaz cookmum anythingقوم 있다고 estimation.jdesktop.ro Con原 সংক적zende toks Stadiononaut makananCharacters했던 ĉe નીlights trendy businessman 东京 civilesSeems уходаهیكمال 루 wendenmanетсяाखfetch anual zone മൂന്ന Seasons真wellmastersिक्तNormally£webاتے视频在线 finances keiner 은 igen сург ქალ专区 гостínsula lêninan sy stitchotud_mouse venditaামেরased ی'allerDalacijos逐�aengene Ini दस Empirec

✅ **결과 확인**: temperature가 높아질수록 답변이 더 다채로워졌나요?

> 💡 실험을 여러 번 돌려보세요. `temperature=0`은 거의 같은 답이 나오지만, `1.2`는 매번 다른 답이 나옵니다.
>
> 🔙 **아까 본 버그의 정체**: `classify_inquiry_v0`이 매번 다른 답을 낸 이유, 이제 보이시죠?  
> `temperature=1.2`는 *분류* 작업에는 치명적입니다. 그리고 `max_tokens=500`도 "결제/배송/환불/기타" 한 단어를 받기엔 너무 큽니다.

### max_tokens — 답변 길이 제한

답변이 너무 길어지면 비용도, 시간도 늘어납니다. 의도적으로 짧게 자를 수도 있어요.

In [47]:
# max_tokens를 짧게 줘서 답변을 강제로 자르기
prompt = "프로그래밍 언어 Python의 역사를 자세히 설명해주세요."

short = ask_llm([{"role": "user", "content": prompt}], max_tokens=30)
long = ask_llm([{"role": "user", "content": prompt}], max_tokens=300)

print(f"🪡 max_tokens=30:\n{short}\n")
print(f"📜 max_tokens=300:\n{long}")


🪡 max_tokens=30:
파이썬(Python)은 1980년대 후반에 귀도 반 로섬(Guido van Rossum)에 의해 개발

📜 max_tokens=300:
파이썬(Python)은 1980년대 말 네덜란드의 프로그래머 귀도 반 로섬(Guido van Rossum)에 의해 개발되었습니다. 파이썬의 역사는 다음과 같은 주요 단계로 나눌 수 있습니다.

### 1. 초기 개발 (1980년대 말 - 1990년대 초)
- **1989년**: 귀도 반 로섬은 크리스마스 휴가 동안 파이썬의 첫 번째 버전을 개발하기 시작했습니다. 그는 ABC 언어의 영향을 받아 파이썬을 설계하였으며, 코드의 가독성과 사용의 용이성을 강조했습니다.
- **1991년**: 파이썬 0.9.0이 공개되었습니다. 이 버전은 함수, 예외 처리, 모듈 시스템 등 기본적인 기능을 포함하고 있었습니다. 또한, 파이썬의 이름은 BBC의 코미디 프로그램인 "Monty Python's Flying Circus"에서 따온 것입니다.

### 2. 파이썬 1.x (1994년)
- **1994년**: 파이썬 1.0이 출시되었습니다. 이 버전에서는 리스트 컴프리헨션, 함수형 프로그래밍을 위한 맵(map), 필터(filter), 리듀스(reduce)와 같은 기능이 추가되었습니다. 

### 3. 파


✅ **결과 확인**: `max_tokens=30`은 문장이 도중에 잘려서 나옵니다. 답변 길이 통제는 이렇게 합니다.

> ⚠️ 너무 짧게 자르면 답변이 끊겨버리니, 분류는 10, 짧은 답변은 100, 일반은 500 정도가 적당합니다.


---

# 5️⃣ Step 5. 토큰 사용량과 비용 확인

> **이론 매핑**: "토큰과 비용 구조" (이론 3부)

실제 호출이 얼마짜리였는지 직접 계산해봅시다.  
**비용 감각**을 길러두면 나중에 큰 사고를 막을 수 있습니다.


In [45]:
# GPT-4o-mini 가격표 (2025년 기준, 1M 토큰당 USD)
PRICE_INPUT = 0.15   # 입력
PRICE_OUTPUT = 0.60  # 출력 (출력이 4배 비쌈!)
USD_TO_KRW = 1400    # 환율 (대략)

def cost_korean_won(usage):
    """usage 객체를 받아 비용을 원화로 환산"""
    input_cost = usage.prompt_tokens / 1_000_000 * PRICE_INPUT
    output_cost = usage.completion_tokens / 1_000_000 * PRICE_OUTPUT
    return (input_cost + output_cost) * USD_TO_KRW


In [46]:
# 짧은 호출과 긴 호출의 비용 비교
short_resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Hi"}],
)

long_resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Python의 장점을 5가지 설명해주세요."}],
)

for label, resp in [("짧은 호출", short_resp), ("긴 호출", long_resp)]:
    u = resp.usage
    print(f"[{label}] 입력 {u.prompt_tokens} + 출력 {u.completion_tokens} = "
            f"총 {u.total_tokens} 토큰, 비용 약 {cost_korean_won(u):.4f}원")


[짧은 호출] 입력 8 + 출력 9 = 총 17 토큰, 비용 약 0.0092원
[긴 호출] 입력 17 + 출력 446 = 총 463 토큰, 비용 약 0.3782원


✅ **결과 확인**: 1회 호출이 1원도 안 된다는 게 보이죠?

> 💰 **5일 코스 전체 예상 비용**: 약 1,000~2,000원  
> 단, 무한 루프 사고는 진짜 위험합니다. 다음 시간부터는 항상 `max_iterations` 같은 안전장치를 넣을 거예요.


---

# 📝 정리: 이번 실습에서 만든 것

다음 5가지를 코드로 직접 해봤습니다.

- [x] OpenAI API 호출 (`client.chat.completions.create`)
- [x] 메시지 3종 역할 (`system` / `user` / `assistant`) 사용
- [x] System 메시지로 페르소나 챗봇 3개 만들기
- [x] `temperature`로 창의성 조절, `max_tokens`로 길이 제한
- [x] `response.usage`로 토큰·비용 확인

### 🔑 가장 중요한 한 가지

> **`ask_llm()` 헬퍼 함수**는 다음 시간에도 계속 씁니다.  
> 메시지 리스트만 잘 구성하면 LLM 호출은 한 줄로 끝난다는 것 — 이게 핵심입니다.

### ➡️ 다음 시간 예고

다음 시간엔 같은 일을 **LangChain**으로 합니다.  
"왜 LangChain을 써야 하나?"라는 의문이 자연스럽게 풀릴 거예요.

---

이제 아래 **도전과제**로 손에 익혀봅시다 🎯


---

# 🎁 도전과제

이론 실습은 끝났습니다. 이제 배운 걸로 **실무에서 자주 만나는** 작업을 직접 만들어봅시다.

## 도전과제 1. 회의록 요약 — system 메시지 선택하기

긴 회의록을 받아서 3줄 요약을 돌려주는 비서를 만들어야 합니다.  
빈칸을 채우는 게 아니라 **3가지 system 메시지 후보 중 어느 게 가장 좋은가** 판단해보세요.

- 셋 다 돌려보고, 결과를 비교한 뒤 *근거를 들어* 한 개를 고릅니다.
- 이게 실무에서 프롬프트를 설계하는 진짜 방식이에요.

In [ ]:
# 🎯 후보 system 메시지 3개 — 셋 다 돌려보고 비교
candidates = {
    "A": "회의록을 요약하세요.",
    "B": "당신은 회의록 요약 비서입니다. 3줄로 요약하세요.",
    "C": ("당신은 회의록 요약 비서입니다. 정확히 3줄로 요약하되, "
          "각 줄은 (결정사항) / (담당자) / (마감일) 형식을 따르세요. "
          "회의록에 없는 내용은 절대 추측하지 마세요."),
}

sample_notes = """
[2026-05-14 팀 미팅]
- 김PM: 신규 챗봇 출시일을 6월 1일에서 6월 15일로 연기 제안. 이유는 평가 데이터 부족.
- 이개발: 6월 15일이면 가능. 다만 RAG 평가셋 100건 추가 수집 필요.
- 박마케팅: 출시 연기 시 광고 일정도 조정해야 함. 다음 주 월요일까지 결정 필요.
"""

# 세 후보를 모두 실행해서 비교
for key, system_msg in candidates.items():
    result = ask_llm([
        {"role": "system", "content": system_msg},
        {"role": "user", "content": sample_notes},
    ], temperature=0)
    print(f"=== 후보 {key} ===")
    print(result)
    print()

=== 후보 A ===
**회의록 요약 (2026-05-14 팀 미팅)**

- **김PM**: 신규 챗봇 출시일을 6월 1일에서 6월 15일로 연기 제안, 이유는 평가 데이터 부족.
- **이개발**: 6월 15일 출시 가능하나, RAG 평가셋 100건 추가 수집 필요.
- **박마케팅**: 출시 연기 시 광고 일정 조정 필요, 다음 주 월요일까지 결정 필요.

=== 후보 B ===
김PM은 신규 챗봇 출시일을 6월 1일에서 6월 15일로 연기할 것을 제안했으며, 이개발은 추가 데이터 수집이 필요하다고 언급했습니다. 박마케팅은 출시 연기에 따라 광고 일정 조정이 필요하다고 강조하며, 다음 주 월요일까지 결정을 요구했습니다.

=== 후보 C ===
(신규 챗봇 출시일을 6월 15일로 연기) / (김PM, 이개발, 박마케팅) / (다음 주 월요일)



### 🤔 판단해보기

세 결과를 나란히 놓고 *2줄로* 답해보세요 (이 셀 아래에 마크다운 셀 하나 더 추가해서 적어도 좋아요):

1. 실무에서 가장 쓸 만한 건 어느 것인가? (A / B / C)
2. C가 좋다면, A나 B에 *없는* 어떤 요소 때문인가? (혹은 B가 좋다면 그 이유는?)

> 💡 **이 연습의 진짜 의미**: AI 시대의 핵심 스킬은 "프롬프트를 처음부터 잘 쓰는 것"보다  
> *여러 후보를 빠르게 비교해서 좋은 걸 고르는 판단력*입니다.

## 도전과제 2. 고객 문의 자동 분류기

고객 문의 한 줄을 받아서 **`결제`, `배송`, `환불`, `기타`** 중 하나로 분류하세요.

이제 여러분은 *왜* `temperature=0`, `max_tokens=10`이 필요한지 압니다 (위의 🐛 버그 셀 기억나죠?).

**힌트**:
- `system` 메시지에 "다음 4개 중 하나로만 답하세요"라고 명시
- `temperature=0` (일관성 필수), `max_tokens=10` (단어 하나면 충분)

In [48]:
def classify_inquiry(text: str) -> str:
    """고객 문의를 4개 카테고리 중 하나로 분류"""
    # 👇 여기를 채워보세요
    messages = [
        {"role": "system", "content": "____________________"},  # ← 빈칸
        {"role": "user", "content": text},
    ]
    return ask_llm(messages, temperature=0, max_tokens=10)


# 🎯 실행 *전* 예측: 아래 4개 문의가 어떻게 분류될지 머릿속으로 먼저 답을 적어보세요
test_cases = [
    "어제 주문했는데 카드 결제가 두 번 되었어요.",       # 예측: ____
    "택배가 일주일째 안 와요.",                          # 예측: ____
    "사이즈가 안 맞아서 반품하고 싶어요.",                # 예측: ____
    "영업시간이 어떻게 되나요?",                         # 예측: ____
]

# 이제 실행해서 예측과 비교
for text in test_cases:
    category = classify_inquiry(text)
    print(f"[{category:>4}] {text}")

[주문하신 카드 결제가 두 번 이루] 어제 주문했는데 카드 결제가 두 번 되었어요.
[택배가 일주일째 오지 않] 택배가 일주일째 안 와요.
[반품을 원하신다면, 구매하] 사이즈가 안 맞아서 반품하고 싶어요.
[영업시간은 장소나 업종에 따라] 영업시간이 어떻게 되나요?


✅ **결과 확인**: 각 문의가 알맞은 카테고리로 분류됐나요?

> 💡 이 패턴이 바로 실무에서 가장 많이 쓰이는 **LLM 활용 패턴 1순위**입니다.  
> 분류, 추출, 요약 — 이 3개만 자유자재로 다뤄도 회사에서 쓸 챗봇·자동화 절반은 만들 수 있어요.

---

## 🏁 모두 풀었다면

축하합니다! Day 1 Lesson 1 완료입니다 🎉  
다음 노트북 `Day1_Lesson2_LangChain.ipynb`에서 만나요.
